# 📊 Komparasi Arsitektur Deep Learning: Simple RNN vs GRU pada Analisis Sentimen Teks Bahasa Indonesia

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---
### 📌 Informasi Tugas
- **Mata Kuliah:** Deep Learning
- **Topik:** Implementasi dan Komparasi Arsitektur Sekuensial (Simple RNN vs GRU)
- **Dataset:** IndoNLU SmSA (Sentiment Analysis Prosa Bahasa Indonesia)
- **Framework:** TensorFlow / Keras 3

### 🎯 Tujuan Eksperimen
1. Membangun model pemrosesan teks sekuensial dengan arsitektur **Simple RNN** dan **GRU**.
2. Melakukan benchmarking performa: **Accuracy, Macro F1-Score, Loss, dan Waktu Pelatihan**.
3. Menganalisis pengaruh mekanisme gating pada GRU terhadap mitigasi problem *vanishing gradient*.


## 1. Import Library & Konfigurasi Lingkungan

In [ ]:
import os
import re
import time
import string
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from keras import layers
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

sns.set_theme(style='whitegrid')
print('TensorFlow version:', tf.__version__)
print('Keras version:', keras.__version__)

## 2. Memuat Dataset IndoNLU SmSA
Dataset SmSA dari IndoNLU terdiri dari 11.000 data latih, 1.260 data validasi, dan 500 data uji.

In [ ]:
base_url = 'https://raw.githubusercontent.com/indobenchmark/indonlu/master/dataset/smsa_doc-sentiment-prosa/'
train_url = base_url + 'train_preprocess.tsv'
valid_url = base_url + 'valid_preprocess.tsv'
test_url  = base_url + 'test_preprocess.tsv'

df_train = pd.read_csv(train_url, sep='\t', header=None, names=['text', 'label'])
df_valid = pd.read_csv(valid_url, sep='\t', header=None, names=['text', 'label'])
df_test  = pd.read_csv(test_url,  sep='\t', header=None, names=['text', 'label'])

print(f'Train samples : {len(df_train)}')
print(f'Valid samples : {len(df_valid)}')
print(f'Test samples  : {len(df_test)}')
df_train.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.countplot(data=df_train, x='label', palette='mako', ax=axes[0])
axes[0].set_title('Distribusi Kelas Sentimen (Data Latih)', fontweight='bold')
axes[0].set_xlabel('Sentimen')
axes[0].set_ylabel('Jumlah')

df_train['word_len'] = df_train['text'].apply(lambda x: len(str(x).split()))
sns.histplot(df_train['word_len'], bins=30, kde=True, color='teal', ax=axes[1])
axes[1].set_title('Distribusi Jumlah Kata per Ulasan', fontweight='bold')
axes[1].set_xlabel('Jumlah Kata')
plt.tight_layout()
plt.show()

print('Statistik Kata:', df_train['word_len'].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

## 4. Text Preprocessing & Vectorization

In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'https?:\/\/\S+|www\.\S+', '', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_train['clean'] = df_train['text'].apply(clean_text)
df_valid['clean'] = df_valid['text'].apply(clean_text)
df_test['clean']  = df_test['text'].apply(clean_text)

LABEL_MAP = {'negative': 0, 'neutral': 1, 'positive': 2}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

y_train = df_train['label'].map(LABEL_MAP).values
y_valid = df_valid['label'].map(LABEL_MAP).values
y_test  = df_test['label'].map(LABEL_MAP).values

VOCAB_SIZE = 8000
MAX_LEN = 80
EMBEDDING_DIM = 64

vectorizer = layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN,
    standardize=None
)
vectorizer.adapt(df_train['clean'].values)

X_train = vectorizer(df_train['clean'].values).numpy()
X_valid = vectorizer(df_valid['clean'].values).numpy()
X_test  = vectorizer(df_test['clean'].values).numpy()

print('Bentuk Matriks Token Latih:', X_train.shape)

## 5. Model 1: Simple RNN (Recurrent Neural Network)
Arsitektur *recurrent* konvensional dengan *weight sharing* di setiap *time-step*.

In [ ]:
model_rnn = keras.Sequential([
    layers.Input(shape=(MAX_LEN,), name='input_ids'),
    layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM, mask_zero=True, name='embedding'),
    layers.SpatialDropout1D(0.3, name='spatial_dropout'),
    layers.SimpleRNN(64, dropout=0.3, return_sequences=False, name='simple_rnn'),
    layers.Dense(32, activation='relu', name='dense_hidden'),
    layers.Dropout(0.3, name='dropout'),
    layers.Dense(3, activation='softmax', name='output_layer')
], name='Simple_RNN_Model')

model_rnn.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_rnn.summary()

In [ ]:
callbacks_rnn = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
]

t0 = time.time()
history_rnn = model_rnn.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=8,
    batch_size=64,
    callbacks=callbacks_rnn,
    verbose=1
)
time_rnn = time.time() - t0
print(f'Waktu Pelatihan Simple RNN: {time_rnn:.2f} detik')

## 6. Model 2: GRU (Gated Recurrent Unit)
Arsitektur *recurrent* dengan Reset Gate dan Update Gate untuk mempertahankan memori jangka panjang.

In [ ]:
model_gru = keras.Sequential([
    layers.Input(shape=(MAX_LEN,), name='input_ids'),
    layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM, mask_zero=True, name='embedding'),
    layers.SpatialDropout1D(0.3, name='spatial_dropout'),
    layers.GRU(64, dropout=0.3, return_sequences=False, name='gru'),
    layers.Dense(32, activation='relu', name='dense_hidden'),
    layers.Dropout(0.3, name='dropout'),
    layers.Dense(3, activation='softmax', name='output_layer')
], name='GRU_Model')

model_gru.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_gru.summary()

In [ ]:
callbacks_gru = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
]

t0 = time.time()
history_gru = model_gru.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=8,
    batch_size=64,
    callbacks=callbacks_gru,
    verbose=1
)
time_gru = time.time() - t0
print(f'Waktu Pelatihan GRU: {time_gru:.2f} detik')

## 7. Evaluasi & Komparasi Kinerja pada Test Set

In [ ]:
target_names = ['Negatif', 'Netral', 'Positif']

y_pred_rnn = np.argmax(model_rnn.predict(X_test, verbose=0), axis=1)
y_pred_gru = np.argmax(model_gru.predict(X_test, verbose=0), axis=1)

print('=== Classification Report: Simple RNN ===')
print(classification_report(y_test, y_pred_rnn, target_names=target_names))

print('=== Classification Report: GRU ===')
print(classification_report(y_test, y_pred_gru, target_names=target_names))

df_results = pd.DataFrame([
    {
        'Model': 'Simple RNN',
        'Akurasi': f'{accuracy_score(y_test, y_pred_rnn)*100:.2f}%',
        'Macro F1': f'{f1_score(y_test, y_pred_rnn, average="macro"):.4f}',
        'Training Time (s)': f'{time_rnn:.2f}'
    },
    {
        'Model': 'GRU',
        'Akurasi': f'{accuracy_score(y_test, y_pred_gru)*100:.2f}%',
        'Macro F1': f'{f1_score(y_test, y_pred_gru, average="macro"):.4f}',
        'Training Time (s)': f'{time_gru:.2f}'
    }
])
df_results

### Visualisasi Confusion Matrix & Kurva Belajar

In [ ]:
cm_rnn = confusion_matrix(y_test, y_pred_rnn)
cm_gru = confusion_matrix(y_test, y_pred_gru)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(cm_rnn, annot=True, fmt='d', cmap='Reds', xticklabels=target_names, yticklabels=target_names, ax=axes[0])
axes[0].set_title('Confusion Matrix - Simple RNN', fontweight='bold')
axes[0].set_xlabel('Prediksi')
axes[0].set_ylabel('Aktual')

sns.heatmap(cm_gru, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names, ax=axes[1])
axes[1].set_title('Confusion Matrix - GRU', fontweight='bold')
axes[1].set_xlabel('Prediksi')
axes[1].set_ylabel('Aktual')
plt.tight_layout()
plt.show()

## 8. Uji Coba Inferensi Teks Kustom

In [ ]:
test_sentences = [
    'Makanannya sangat lezat dan suasana restorannya sangat menyenangkan!',
    'Pengiriman lambat sekali, barang datang rusak dan pelayanan CS sangat buruk.',
    'Saya memesan dua cangkir kopi susu dan duduk di dekat jendela.'
]

for s in test_sentences:
    clean_s = clean_text(s)
    vec_s = vectorizer([clean_s]).numpy()
    prob_rnn = model_rnn.predict(vec_s, verbose=0)[0]
    prob_gru = model_gru.predict(vec_s, verbose=0)[0]
    
    lbl_rnn = INV_LABEL_MAP[np.argmax(prob_rnn)]
    lbl_gru = INV_LABEL_MAP[np.argmax(prob_gru)]
    
    print(f'Kalimat  : "{s}"')
    print(f'-> Simple RNN : {lbl_rnn.upper()} (P={np.max(prob_rnn):.4f})')
    print(f'-> GRU        : {lbl_gru.upper()} (P={np.max(prob_gru):.4f})\n')

## 9. Kesimpulan Analisis
1. **Keunggulan GRU:** Mekanisme *Reset Gate* dan *Update Gate* pada GRU memungkinkan model memilih informasi mana yang harus dipertahankan dan dilupakan, menghasilkan akurasi yang lebih unggul dibandingkan Simple RNN.
2. **Trade-off Waktu:** Simple RNN memiliki waktu latih lebih cepat karena persamaan matematika yang lebih sederhana tanpa struktur gate, namun memiliki limitasi performa pada kalimat panjang (*vanishing gradient problem*).
3. **Rekomendasi:** Untuk pemodelan sekuensial teks di industri maupun riset lanjutan, varian gated seperti GRU atau LSTM merupakan standar arsitektur RNN yang direkomendasikan.